# Paired Switching Strategy Research Notebook

This notebook replicates the core logic of the Paired Switching algorithm for research and analysis. We will perform:
1. Universe Selection (Coarse and Fine)
2. Stock Metric Calculation (Momentum, Volatility, etc.)
3. Market Regime Clustering
4. Regression and Correlation Analysis

In [ ]:
# Import necessary libraries
from datetime import timedelta, datetime
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import seaborn as sns

# Create a QuantBook instance
qb = QuantBook()

# Set analysis date
qb.SetEndDate(datetime(2025, 3, 15))

## 1. Universe Selection

First, we define the universe of stocks we want to analyze. This follows the same coarse and fine filtering logic as the main algorithm.

In [ ]:
def coarse_filter(coarse):
    """Filter stocks by price and volume."""
    filtered = [x for x in coarse if x.HasFundamentalData and x.Price > 1]
    sorted_by_dollar_vol = sorted(filtered, key=lambda x: x.DollarVolume, reverse=True)
    return [x.Symbol for x in sorted_by_dollar_vol[:1000]]

def fine_filter(fine):
    """Select top 100 stocks by market cap in each sector."""
    sector_dict = {}
    sector_mcap = {}
    
    for f in fine:
        if f.MarketCap == 0: continue
        sector = f.AssetClassification.MorningstarSectorCode
        if sector not in sector_dict:
            sector_dict[sector] = []
            sector_mcap[sector] = 0
        sector_dict[sector].append(f)
        sector_mcap[sector] += f.MarketCap
    
    top_5_sectors = sorted(sector_mcap, key=lambda x: sector_mcap[x], reverse=True)[:5]
    
    selected = []
    for sector in top_5_sectors:
        sorted_sector = sorted(sector_dict[sector], key=lambda x: x.MarketCap, reverse=True)
        selected.extend([x.Symbol for x in sorted_sector[:100]])
    
    return selected

# Add universe and get symbols
qb.AddUniverse(coarse_filter, fine_filter)
universe_symbols = qb.ActiveSecurities.Keys

# Add SPY for baseline comparison
spy_symbol = qb.AddEquity("SPY", Resolution.Daily).Symbol
all_symbols = list(universe_symbols)
if spy_symbol not in all_symbols:
    all_symbols.append(spy_symbol)

print(f"Selected {len(all_symbols)} symbols for analysis.")

## 2. Collect Stock Metrics

Next, we collect historical price and volume data to calculate key metrics like momentum and volatility for each stock in our universe.

In [ ]:
def collect_stock_metrics(symbols):
    """Collect price, momentum, and other metrics for all stocks in universe."""
    if not symbols:
        print("No symbols in universe")
        return None
    
    print(f"Collecting metrics for {len(symbols)} stocks...")
    
    try:
        history = qb.History(symbols, 252, Resolution.Daily)
        if history.empty:
            print("No historical data retrieved")
            return None
        
        metrics = {}
        for symbol_obj in symbols:
            if symbol_obj not in history.index.levels[0]:
                continue
            
            sym_history = history.loc[symbol_obj]
            
            if len(sym_history) < 252 * 0.5 or 'close' not in sym_history.columns:
                continue
                
            close_prices = np.array(sym_history['close'].values, dtype=np.float64)
            
            if 'volume' in sym_history.columns:
                volumes = np.array(sym_history['volume'].values, dtype=np.float64)
                avg_volume = np.mean(volumes)
            else:
                avg_volume = 0
            
            current_price = close_prices[-1]
            start_price = close_prices[0]
            price_change = (current_price - start_price) / start_price if start_price != 0 else 0
            
            if len(close_prices) > 21:
                prev_price = close_prices[-21]
                momentum = (current_price - prev_price) / prev_price if prev_price != 0 else 0
            else:
                momentum = 0
            
            if len(close_prices) > 1:
                prices_t = close_prices[1:]
                prices_t_minus_1 = close_prices[:-1]
                with np.errstate(divide='ignore', invalid='ignore'):
                    returns = (prices_t - prices_t_minus_1) / prices_t_minus_1
                returns = returns[np.isfinite(returns)]
                volatility = np.std(returns) if len(returns) > 0 else 0
            else:
                volatility = 0
            
            metrics[symbol_obj] = {
                'price': current_price,
                'price_change': price_change,
                'momentum': momentum,
                'volatility': volatility,
                'volume': avg_volume
            }
        
        print(f"Collected metrics for {len(metrics)} stocks")
        return pd.DataFrame(metrics).T if len(metrics) > 0 else None
        
    except Exception as e:
        print(f"Error collecting metrics: {str(e)}")
        return None

metrics_df = collect_stock_metrics(all_symbols)
if metrics_df is not None:
    display(metrics_df.head())

## 3. Market Regime Clustering

We classify stocks into one of six market regimes based on their 21-day momentum and daily volatility. This is a rule-based clustering approach.

In [ ]:
def perform_clustering(metrics_df):
    """Classify stocks into 6 Market Regimes based on Momentum and Volatility."""
    if metrics_df is None or len(metrics_df) < 1:
        print("Insufficient data for clustering")
        return None, None
    
    regime_names = [
        "Calm Bull", "Volatile Bull",
        "Calm Bear", "Volatile Bear",
        "Calm Sideways", "Volatile Sideways"
    ]
    groups = {name: [] for name in regime_names}
    group_assignments = {}
    
    for symbol, row in metrics_df.iterrows():
        mom = row['momentum']
        vol = row['volatility']
        
        if mom > 0.02: trend = "Bull"
        elif mom < -0.02: trend = "Bear"
        else: trend = "Sideways"
        
        if vol > 0.015: vol_type = "Volatile"
        else: vol_type = "Calm"
        
        grp = f"{vol_type} {trend}"
        
        if grp in groups:
            groups[grp].append(symbol)
            group_assignments[symbol] = grp
            
    print("Clustering complete.")
    return groups, group_assignments

correlation_groups, group_assignments = perform_clustering(metrics_df)
if correlation_groups:
    # Add the group assignment to the main dataframe for easy filtering
    metrics_df['group'] = metrics_df.index.map(group_assignments)
    display(metrics_df.head())

### Cluster Visualization

Let's visualize the clusters on a scatter plot of Momentum vs. Volatility.

In [ ]:
if metrics_df is not None and 'group' in metrics_df.columns:
    plt.figure(figsize=(12, 8))
    sns.scatterplot(data=metrics_df, x='volatility', y='momentum', hue='group', palette='viridis', s=50, alpha=0.7)
    
    # Add regime boundaries
    plt.axhline(0.02, color='grey', linestyle='--', lw=1)
    plt.axhline(-0.02, color='grey', linestyle='--', lw=1)
    plt.axvline(0.015, color='grey', linestyle='--', lw=1)
    
    plt.title('Stock Clustering: Momentum vs. Volatility')
    plt.xlabel('Daily Volatility (Std. Dev. of Returns)')
    plt.ylabel('21-Day Momentum')
    plt.legend(title='Market Regime')
    plt.grid(True, which='both', linestyle='--', linewidth=0.5)
    plt.show()

## 4. Group Analysis

Now we can analyze the characteristics of each group.

In [ ]:
def generate_group_report(metrics_df, correlation_groups):
    """Generate and print group report."""
    if not correlation_groups:
        print("No groups to report")
        return
    
    print("\n" + "=" * 50)
    print("GROUP REPORT")
    print("=" * 50)
    
    for group_name, symbols in sorted(correlation_groups.items()):
        if not symbols:
            continue
        
        group_metrics = metrics_df.loc[symbols]
        
        print(f"\n--- GROUP: {group_name} ({len(symbols)} members) ---")
        display(group_metrics.describe())

if metrics_df is not None and correlation_groups:
    generate_group_report(metrics_df, correlation_groups)

## 5. Regression Analysis

We perform a simple linear regression to see how well group membership explains the variance in each of our calculated metrics. A high R-squared value suggests the metric is a strong differentiator for that group.

In [ ]:
def perform_regression_analysis(metrics_df):
    """Perform regression analysis between groups and metrics."""
    if 'group' not in metrics_df.columns:
        print("No group assignments for regression analysis")
        return
    
    # Create numeric representation of groups
    group_codes, unique_groups = pd.factorize(metrics_df['group'])
    
    results = {}
    for metric_col in ['price', 'price_change', 'momentum', 'volatility', 'volume']:
        X = group_codes.reshape(-1, 1)
        y = metrics_df[metric_col].values
        
        model = LinearRegression()
        model.fit(X, y)
        r_squared = model.score(X, y)
        results[metric_col] = r_squared
    
    print("\n" + "=" * 60)
    print("REGRESSION ANALYSIS REPORT - Group vs. Metrics Correlation")
    print("=" * 60)
    print("R-squared values indicate how well group membership explains each metric:")
    
    sorted_results = sorted(results.items(), key=lambda x: x[1], reverse=True)
    for metric, r_squared in sorted_results:
        print(f"  - {metric:<20} R² = {r_squared:.4f}")

if metrics_df is not None:
    perform_regression_analysis(metrics_df)

## 6. Inter-Group Correlation Analysis

Finally, we analyze the correlation *between* the different market regime groups. We create an average daily return series for each group and then calculate the correlation matrix. This helps us understand which groups move together and which diverge.

In [ ]:
def analyze_group_correlations(correlation_groups):
    """Calculate and visualize correlations between groups based on daily returns."""
    if not correlation_groups:
        return

    all_symbols = [spy_symbol]
    for symbols in correlation_groups.values():
        all_symbols.extend(symbols)
    all_symbols = list(set(all_symbols)) # Unique symbols
    
    history = qb.History(all_symbols, 63, Resolution.Daily)
    if history.empty:
        return

    closes = history['close'].unstack(level=0)
    returns = closes.pct_change().dropna()

    group_series = {}
    # Add SPY as baseline
    if spy_symbol in returns.columns:
        group_series['SPY'] = returns[spy_symbol]
        
    for group_name, symbols in correlation_groups.items():
        group_syms = [s for s in symbols if s in returns.columns]
        if not group_syms:
            continue
        group_series[group_name] = returns[group_syms].mean(axis=1)

    if not group_series:
        print("Could not calculate group series for correlation.")
        return

    group_df = pd.DataFrame(group_series)
    corr_matrix = group_df.corr()

    print("\n" + "=" * 50)
    print("INTER-GROUP CORRELATION MATRIX (Daily Returns)")
    print("=" * 50)
    display(corr_matrix)

    # Plot heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
    plt.title('Inter-Group Correlation of Daily Returns')
    plt.show()

if correlation_groups:
    analyze_group_correlations(correlation_groups)